In [ ]:
import polars as pl
import json
from collections import defaultdict
from scipy.spatial.distance import jensenshannon
import numpy as np
import itertools

In [ ]:
df = pl.read_csv("../dataset/Capstone2025_nsi_lvl9_with_landcover_and_color.csv.gz")

In [ ]:
df

In [ ]:
df["clr"].n_unique()

In [ ]:
df = pl.read_csv("../dataset/Capstone2025_nsi_lvl9_with_landcover_and_color.csv.gz")

with open("../dataset/ca-county-neighbors.json") as f:
    neighbors_raw = json.load(f)

neighbors = pl.DataFrame(neighbors_raw).with_columns(
    pl.col("county_fips").cast(pl.String).str.zfill(5),
    pl.col("neighbor_fips").cast(pl.String).str.zfill(5),
)

df= df.with_columns(pl.col("fips").cast(pl.String).str.zfill(5))


In [ ]:
df

In [ ]:
# sum clr_cc column


# Manual

In [ ]:
from jsd_calc import compute_neighbor_jsd

LAPLACE_PSEUDOCOUNT = 1
MIN_SUPPORT = 30
semantic_groups = {
    "blue": ["aqua", "aquamarine", "azure", "blue", "navy", "bar"],
    "purple": ["indigo", "lavender", "lilac", "plum", "purple"],
    "red": ["auburn", "crimson", "maroon", "red", "scarlet", "foo"],
    "orange": ["orange", "sienna", "terracotta"],
    "yellow": ["amber", "gold", "lemon", "yellow"],
    "green": ["emerald", "green", "olive", "sage", "verde"],
    "brown": ["beige", "brown", "cocoa", "coffee", "tan"],
    "gray": ["alabaster", "gray", "grey", "ivory"],
}

semantic_map = {
    color: canon
    for canon, members in semantic_groups.items()
    for color in members
}

all_colors_raw = df["clr"].unique().sort().to_list()
unmapped_colors = sorted([c for c in all_colors_raw if c not in semantic_map])
if unmapped_colors:
    print(f"Unmapped colors (kept as single colors): {unmapped_colors}")

jsd_results_semantic, jsd_stats_semantic = compute_neighbor_jsd(
    df=df,
    neighbors=neighbors,
    final_map=semantic_map,
    laplace_pseudocount=float(LAPLACE_PSEUDOCOUNT),
    min_support=int(MIN_SUPPORT),
)

summary_semantic = {
    "n_pairs": int(jsd_stats_semantic["total_pairs"]),
    "n_semantic_colors": int(jsd_stats_semantic["n_merged_colors"]),
    "mean_weighted_jsd": float(jsd_stats_semantic["mean_jsd"]),
    "mean_unweighted_jsd": float(jsd_results_semantic["mean_jsd"].mean()) if jsd_results_semantic.height else 0.0,
}
summary_semantic

manual best is 0.184

# Config

In [ ]:
strata_cols = ["st_damcat", "bldgtype", "lc_type"]
TARGET_MIN_ROWS = 100
AVG_BLDG_PER_ROW = df["clr_cc"].sum() / df.height
MIN_TOTAL = int(round(TARGET_MIN_ROWS * AVG_BLDG_PER_ROW))
MIN_ROWS = TARGET_MIN_ROWS

MIN_VOTES = 30
MAX_ROUNDS = 50
STOP_RATIO = 0.12
TOP_K_EXCLUSIVE = 2
RANK_DECAY = 0.5
MAX_MERGES_PER_ROUND = 6

MIN_TOTAL, MIN_ROWS, MIN_VOTES, MAX_ROUNDS

# Static inputs and round state

In [ ]:
color_totals = df.group_by("clr").agg(pl.col("clr_cc").sum().alias("total"))
color_total_map = dict(zip(color_totals["clr"].to_list(), color_totals["total"].to_list()))

current_df = df.clone()
master_map = {}
round1_votes = None

for clr in list(color_total_map.keys())[:5]:
    print(clr, color_total_map[clr])

# Neighbor pair prep

In [ ]:
nb_pairs = []
seen = set()
for row in neighbors.iter_rows(named=True):
    c1, c2 = row["county_fips"], row["neighbor_fips"]
    key = (min(c1, c2), max(c1, c2))
    if key not in seen:
        seen.add(key)
        nb_pairs.append(key)

nb_pairs[:3]

# Run iterative merge loop

In [ ]:
for round_num in range(1, MAX_ROUNDS + 1):
    cc = current_df.group_by(["fips"] + strata_cols + ["clr"]).agg(
        pl.col("clr_cc").sum().alias("count"),
        pl.len().alias("n_rows"),
    )
    gt = cc.group_by(["fips"] + strata_cols).agg(
        pl.col("count").sum().alias("total"),
        pl.col("n_rows").sum().alias("total_rows"),
    )
    cf = (
        cc.join(
            gt.filter((pl.col("total") >= MIN_TOTAL) & (pl.col("total_rows") >= MIN_ROWS)),
            on=["fips"] + strata_cols,
        )
        .with_columns((pl.col("count") / pl.col("total")).alias("freq"))
    )

    if cf.height == 0:
        print(f"Round {round_num}: no eligible strata after minimum thresholds")
        break

    strata_rows = cf.select(["fips"] + strata_cols).unique(maintain_order=True)
    strata_by_fips = defaultdict(list)
    for row in strata_rows.iter_rows(named=True):
        strata_key = tuple(row[c] for c in strata_cols)
        strata_by_fips[row["fips"]].append(strata_key)

    strata_cache = {}
    for fips, strata_list in strata_by_fips.items():
        for st_damcat, bldgtype, lc_type in strata_list:
            data = (
                cf.filter(
                    (pl.col("fips") == fips)
                    & (pl.col("st_damcat") == st_damcat)
                    & (pl.col("bldgtype") == bldgtype)
                    & (pl.col("lc_type") == lc_type)
                )
                .sort("freq", descending=True)
            )
            if data.height == 0:
                continue
            strata_cache[(fips, st_damcat, bldgtype, lc_type)] = (
                data["clr"].to_list(),
                data["freq"][0],
                data.height,
            )

    pair_votes = defaultdict(float)
    for c1, c2 in nb_pairs:
        c1_strata = strata_by_fips.get(c1, [])
        if not c1_strata:
            continue

        for strata_key in c1_strata:
            c1_info = strata_cache.get((c1,) + strata_key)
            c2_info = strata_cache.get((c2,) + strata_key)
            if c1_info is None or c2_info is None:
                continue

            c1_ranked, c1_top_freq, c1_n_colors = c1_info
            c2_ranked, c2_top_freq, c2_n_colors = c2_info

            if c1_top_freq < 1.5 / c1_n_colors or c2_top_freq < 1.5 / c2_n_colors:
                continue

            shared = set(c1_ranked) & set(c2_ranked)
            c1_excl = [c for c in c1_ranked if c not in shared]
            c2_excl = [c for c in c2_ranked if c not in shared]

            k = min(TOP_K_EXCLUSIVE, len(c1_excl), len(c2_excl))
            for i in range(k):
                a, b = min(c1_excl[i], c2_excl[i]), max(c1_excl[i], c2_excl[i])
                pair_votes[(a, b)] += (RANK_DECAY ** i)

    candidates = sorted(
        [(pair, v) for pair, v in pair_votes.items() if v >= MIN_VOTES],
        key=lambda x: x[1],
        reverse=True,
    )
    if not candidates:
        print(f"Round {round_num}: no pairs with >= {MIN_VOTES:g} weighted votes")
        break

    top_votes = candidates[0][1]
    if round_num == 1:
        round1_votes = top_votes
        print(f"Round 1 baseline: {round1_votes:.1f} votes, will stop below {round1_votes * STOP_RATIO:.1f}")

    if top_votes < round1_votes * STOP_RATIO:
        print(
            f"Round {round_num}: top votes ({top_votes:.1f}) < {STOP_RATIO:.0%} of round 1 ({round1_votes:.1f}) - stopping"
        )
        break

    claimed = set()
    round_merges = []
    for (a, b), votes in candidates:
        if len(round_merges) >= MAX_MERGES_PER_ROUND:
            break
        if a in claimed or b in claimed:
            continue

        freq_a = color_total_map.get(a, 0)
        freq_b = color_total_map.get(b, 0)
        canonical, merged = (a, b) if freq_a >= freq_b else (b, a)
        round_merges.append((merged, canonical, votes))
        claimed.add(a)
        claimed.add(b)

    if not round_merges:
        print(f"Round {round_num}: no non-overlapping pairs to merge")
        break

    print(
        f"Round {round_num}: {len(round_merges)} merges (top vote {top_votes:.1f}, {top_votes / round1_votes:.0%} of baseline)"
    )
    for merged, canonical, votes in round_merges:
        print(f"  {merged} -> {canonical} ({votes:.1f} weighted votes)")

    round_map = {merged: canonical for merged, canonical, _ in round_merges}
    for merged, canonical in round_map.items():
        master_map[merged] = canonical
        for k, v in list(master_map.items()):
            if v == merged:
                master_map[k] = canonical

    current_df = current_df.with_columns(pl.col("clr").replace(round_map).alias("clr"))

len(master_map), list(master_map.items())[:10]


# Build final_map from merged groups

In [ ]:
groups = {}
for orig in master_map:
    canon = orig
    while canon in master_map and master_map[canon] != canon:
        canon = master_map[canon]
    groups.setdefault(canon, set()).add(orig)

for canon in list(groups.keys()):
    groups[canon].add(canon)

final_map = {}
print("--- Final groups ---")
for canon in sorted(groups, key=lambda c: -len(groups[c])):
    members = sorted(groups[canon])
    best = max(members, key=lambda c: color_total_map.get(c, 0))
    for m in members:
        final_map[m] = best
    print(f"  {best} <- {members}")

merged_colors = set(final_map.keys())
all_colors_list = sorted(df["clr"].unique().to_list())
single_colors = [c for c in all_colors_list if c not in merged_colors]
print(f"Single Colors ({len(single_colors)}): {single_colors}")

# JSD for greedy pooled colors

In [ ]:
from jsd_calc import compute_neighbor_jsd

jsd_results_df, jsd_stats_greedy = compute_neighbor_jsd(
    df=df,
    neighbors=neighbors,
    final_map=final_map,
    laplace_pseudocount=1.0,
    min_support=30,
)

jsd_stats_greedy


# Plots

In [ ]:
import geopandas as gpd
import requests
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import rcParams
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Helvetica Neue', 'Arial', 'DejaVu Sans']
jsd_results_ungrouped, jsd_stats_ungrouped = compute_neighbor_jsd(
    df=df,
    neighbors=neighbors,
    final_map={},
    laplace_pseudocount=1.0,
    min_support=30,
)
print(f"Ungrouped mean JSD : {jsd_stats_ungrouped['mean_jsd']:.4f}")
print(f"Grouped   mean JSD : {jsd_stats_greedy['mean_jsd']:.4f}")

def county_mean(jsd_df):
    totals, counts = {}, {}
    for row in jsd_df.iter_rows(named=True):
        for k in ("fips_a", "fips_b"):
            fips = row[k]
            totals[fips] = totals.get(fips, 0.0) + row["weighted_jsd"]
            counts[fips] = counts.get(fips, 0) + 1
    return {fips: totals[fips] / counts[fips] for fips in totals}

mean_before = county_mean(jsd_results_ungrouped)
mean_after  = county_mean(jsd_results_df)

GEOJSON_URL = "https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/california-counties.geojson"
resp = requests.get(GEOJSON_URL, timeout=30)
gdf  = gpd.GeoDataFrame.from_features(resp.json()["features"], crs="EPSG:4326")

FIPS_LOOKUP = {
    "Alameda": "06001", "Alpine": "06003", "Amador": "06005", "Butte": "06007",
    "Calaveras": "06009", "Colusa": "06011", "Contra Costa": "06013",
    "Del Norte": "06015", "El Dorado": "06017", "Fresno": "06019",
    "Glenn": "06021", "Humboldt": "06023", "Imperial": "06025", "Inyo": "06027",
    "Kern": "06029", "Kings": "06031", "Lake": "06033", "Lassen": "06035",
    "Los Angeles": "06037", "Madera": "06039", "Marin": "06041",
    "Mariposa": "06043", "Mendocino": "06045", "Merced": "06047",
    "Modoc": "06049", "Mono": "06051", "Monterey": "06053", "Napa": "06055",
    "Nevada": "06057", "Orange": "06059", "Placer": "06061", "Plumas": "06063",
    "Riverside": "06065", "Sacramento": "06067", "San Benito": "06069",
    "San Bernardino": "06071", "San Diego": "06073", "San Francisco": "06075",
    "San Joaquin": "06077", "San Luis Obispo": "06079", "San Mateo": "06081",
    "Santa Barbara": "06083", "Santa Clara": "06085", "Santa Cruz": "06087",
    "Shasta": "06089", "Sierra": "06091", "Siskiyou": "06093", "Solano": "06095",
    "Sonoma": "06097", "Stanislaus": "06099", "Sutter": "06101", "Tehama": "06103",
    "Trinity": "06105", "Tulare": "06107", "Tuolumne": "06109",
    "Ventura": "06111", "Yolo": "06113", "Yuba": "06115",
}

name_col = "name" if "name" in gdf.columns else gdf.columns[0]
gdf["fips"]           = gdf[name_col].map(FIPS_LOOKUP)
gdf["mean_jsd_before"] = gdf["fips"].map(mean_before)
gdf["mean_jsd_after"]  = gdf["fips"].map(mean_after)

# ── 4. Plot ───────────────────────────────────────────────────────────────────
CMAP       = "YlOrRd"
VMIN, VMAX = 0.0, 1.0
BORDER     = "#ffffff"
MISSING    = "#e8e8e8"

fig, axes = plt.subplots(1, 2, figsize=(18, 9))
fig.patch.set_facecolor("white")

titles    = ["Before Pooling", "After Greedy Pooling"]
cols      = ["mean_jsd_before", "mean_jsd_after"]
subtitles = [
    f"Mean neighbor JSD = {jsd_stats_ungrouped['mean_jsd']:.3f}",
    f"Mean neighbor JSD = {jsd_stats_greedy['mean_jsd']:.3f}",
]

norm = Normalize(vmin=VMIN, vmax=VMAX)
sm   = ScalarMappable(cmap=CMAP, norm=norm)
sm.set_array([])

for ax, title, col, sub in zip(axes, titles, cols, subtitles):
    ax.set_facecolor("white")

    gdf[gdf[col].isna()].plot(ax=ax, color=MISSING, edgecolor=BORDER, linewidth=0.4, aspect=None)
    gdf[gdf[col].notna()].plot(
        ax=ax, column=col, cmap=CMAP, vmin=VMIN, vmax=VMAX,
        edgecolor=BORDER, linewidth=0.4, legend=False, aspect=None,
    )

    ax.set_title(title, fontsize=15, fontweight="bold", color="#222222", pad=10)
    ax.text(0.5, -0.02, sub, transform=ax.transAxes,
            ha="center", fontsize=11, color="#555555")
    ax.axis("off")

# shared colorbar
cbar_ax = fig.add_axes([0.92, 0.2, 0.015, 0.6])
cb = fig.colorbar(sm, cax=cbar_ax)
cb.set_label("Mean Neighbor JSD", fontsize=10, color="#444444")
cb.ax.yaxis.set_tick_params(color="#888888", labelsize=9)
for spine in cb.ax.spines.values():
    spine.set_edgecolor("#cccccc")

fig.suptitle("County-Level Mean Neighbor JSD — Before vs. After Color Pooling",
             fontsize=16, fontweight="bold", color="#222222", y=1.01)

plt.tight_layout(rect=[0, 0, 0.91, 1])
plt.savefig("neighbor_jsd_comparison.pdf", format="pdf", bbox_inches="tight", facecolor="white")
plt.savefig("neighbor_jsd_comparison.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved → neighbor_jsd_comparison.pdf / .png")


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Helvetica Neue', 'Arial', 'DejaVu Sans']

label_hex = {
    'red': '#c0392b', 'crimson': '#dc143c', 'scarlet': '#e8240a', 'foo': '#e07070',
    'azure': '#2e6fdc', 'blue': '#2980b9', 'indigo': '#5b2d9e', 'purple': '#7d3fb5',
    'navy': '#1a3a7a', 'lavender': '#9370b8', 'lilac': '#b08ccc',
    'aqua': '#0097a7', 'aquamarine': '#3db896',
    'cocoa': '#8c5c3e', 'brown': '#7b3a1e', 'coffee': '#6b4226', 'beige': '#b89870',
    'olive': '#6b7a00', 'green': '#1e9a50', 'sage': '#7d9060', 'verde': '#2e8a55',
    'alabaster': '#9a9a90', 'gray': '#808080', 'grey': '#6e6e6e', 'ivory': '#a0a08a',
    'amber': '#d4a000', 'gold': '#c8a800', 'lemon': '#c0b800', 'yellow': '#c8a800',
    'orange': '#c86010', 'sienna': '#9a4820', 'terracotta': '#b85c38',
}

group_col = {
    'red':       '#c0392b',
    'navy':      '#2056b8',
    'cocoa':     '#8c5c3e',
    'olive':     '#5a7200',
    'alabaster': '#707070',
    'amber':     '#b88c00',
    'orange':    '#b85a10',
}

class Node:
    def __init__(self, label, left=None, right=None, round_num=0, votes=None):
        self.label, self.left, self.right = label, left, right
        self.round_num, self.votes = round_num, votes
        self.x = None
        self._leaves = None

    def is_leaf(self): return self.left is None

    def get_leaves(self):
        if self._leaves is None:
            self._leaves = [self.label] if self.is_leaf() else \
                           self.left.get_leaves() + self.right.get_leaves()
        return self._leaves

def find_leaf(node, name):
    if node.is_leaf(): return node if node.label == name else None
    return find_leaf(node.left, name) or find_leaf(node.right, name)

def iter_internal(node):
    if not node.is_leaf():
        yield node
        yield from iter_internal(node.left)
        yield from iter_internal(node.right)

def layout(node, x0, x1):
    if node.is_leaf(): node.x = (x0 + x1) / 2; return
    nl, nr = len(node.left.get_leaves()), len(node.right.get_leaves())
    mid = x0 + (x1 - x0) * nl / (nl + nr)
    layout(node.left, x0, mid); layout(node.right, mid, x1)
    node.x = (node.left.x + node.right.x) / 2

merges = [
    (1,'cocoa','brown',371.5),(1,'orange','terracotta',93.0),
    (1,'green','sage',48.5),(1,'azure','blue',46.0),
    (2,'cocoa','coffee',212.0),(2,'olive','green',95.5),
    (2,'navy','lavender',87.0),(2,'azure','indigo',47.0),(2,'red','foo',30.5),
    (3,'purple','azure',163.5),(3,'cocoa','beige',93.0),
    (3,'alabaster','gray',53.5),(3,'olive','verde',50.0),
    (3,'navy','lilac',36.0),(3,'orange','sienna',32.5),
    (4,'alabaster','grey',81.0),(4,'amber','lemon',40.5),
    (4,'navy','aqua',37.0),(4,'red','purple',31.0),
    (5,'alabaster','ivory',156.0),(5,'amber','yellow',70.5),
    (5,'red','crimson',58.0),(5,'navy','aquamarine',44.0),
    (6,'amber','gold',65.0),(6,'red','scarlet',40.0),
]

cluster = {c: Node(c) for c in label_hex}
for rnd, canon, absorb, votes in merges:
    cluster[canon] = Node(canon, cluster[canon], cluster.pop(absorb), rnd, votes)

GROUPS = ['red','navy','cocoa','olive','alabaster','amber','orange']
GAP = 1.8

x_cur, offsets = 0, {}
for g in GROUPS:
    root = cluster[g]
    n = len(root.get_leaves())
    layout(root, 0, n)
    offsets[g] = x_cur
    x_cur += n + GAP

fig, ax = plt.subplots(figsize=(80, 30))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

for r in range(1, 7):
    ax.axhline(r, color='#eeeeee', lw=0.8, zorder=0)

def draw_tree(node, col, xoff):
    if node.is_leaf(): return
    y  = node.round_num
    yl = node.left.round_num  if not node.left.is_leaf()  else 0
    yr = node.right.round_num if not node.right.is_leaf() else 0
    lx = node.left.x + xoff
    rx = node.right.x + xoff
    kw = dict(color=col, lw=12, solid_capstyle='round', zorder=4)
    ax.plot([lx, lx], [yl, y], **kw)
    ax.plot([rx, rx], [yr, y], **kw)
    ax.plot([lx, rx], [y,  y], **kw)
    ax.scatter([(lx+rx)/2], [y], color='white', s=400, zorder=5, edgecolors=col, lw=4)
    ax.text((lx+rx)/2, y, f'{node.votes:.0f}',
            ha='center', va='center', fontsize=48, color=col,
            fontweight='bold', zorder=8,
            bbox=dict(boxstyle='round,pad=0.18', fc='white', ec='none'))
    draw_tree(node.left,  col, xoff)
    draw_tree(node.right, col, xoff)

for g in GROUPS:
    root = cluster[g]; col = group_col[g]; xoff = offsets[g]
    draw_tree(root, col, xoff)

    for lf in root.get_leaves():
        ln = find_leaf(root, lf)
        lx = ln.x + xoff
        c  = label_hex[lf]
        ax.plot([lx, lx], [0, -0.12], color='#cccccc', lw=0.9, zorder=3)
        ax.scatter([lx], [0], color=c, s=1200, zorder=6, edgecolors='white', linewidths=4.0)
        ax.text(lx, -0.18, lf, ha='right', va='top', fontsize=56,
                rotation=48, color='#333333', fontstyle='italic')

    max_r = max(nd.round_num for nd in iter_internal(root))
    mid_x = root.x + xoff
    ax.text(mid_x, max_r + 0.38, g.upper(),
            ha='center', va='bottom', fontsize=72, color=col, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.35', fc='white', ec=col, lw=1.2))

ax.set_yticks(range(1, 7))
ax.set_yticklabels([f'Round {r}' for r in range(1, 7)], color='#555555', fontsize=60)
ax.tick_params(axis='y', length=0, pad=8)
ax.tick_params(axis='x', bottom=False, labelbottom=False)
ax.set_ylim(-1.7, 7.3)
ax.set_xlim(-1.2, x_cur - GAP + 0.8)
ax.axhline(0, color='#dddddd', lw=0.8, zorder=1)

for sp in ['top', 'right', 'bottom']: ax.spines[sp].set_visible(False)
ax.spines['left'].set_color('#cccccc')

ax.set_title('Greedy Color Pooling — Merge Dendrogram',
             fontsize=96, fontweight='bold', color='#222222', pad=16)

plt.tight_layout()
plt.savefig('color_pool_dendrogram.pdf', format='pdf', bbox_inches='tight', facecolor='white')
plt.savefig('color_pool_dendrogram.png', dpi=600, bbox_inches='tight', facecolor='white')
plt.show()
print("Saved → color_pool_dendrogram.pdf  (vector, poster-ready)")
print("Saved → color_pool_dendrogram.png  (600 dpi raster)")
